<div style="border:solid green 2px; padding: 20px"> <h1 style="color:green; margin-bottom:20px">Reviewers comment v1</h1>

Hello Obed!

Im happy to review your project today 🙌

My name is **Gerardo Flores** and you can find me on the HUB as https://hub.tripleten.com/u/1949959f

You can find my comments under the heading **«Review»**. I will categorize my comments in green, blue or red boxes like this:

<div class="alert alert-success">
    <b>Success:</b> if everything is done successfully
</div>
<div class="alert alert-warning">
    <b>Remarks:</b> if I can give some recommendations or ways to improve the project
</div>
<div class="alert alert-danger">
    <b>Needs fixing:</b> if the block requires some corrections. Work cant be accepted with the red comments
</div>

Please dont remove my comments :) If you have any questions dont hesitate to respond to my comments in a different section. 
<div class="alert alert-info"> <b>Student comments:</b> For example like this</div>    


<div style="border:solid green 2px; padding: 20px">
<b>Reviewer's comment v1:</b>

    
<b>Overall Feedback</b>
    
Hello Obed,
    
You’ve submitted another project—great work! Your commitment to pushing through the challenges of this program is admirable.

After reviewing your submission, it is approved.
    
   
You can find my more detailed notes within your project notebook in the `Reviewer's comment v1:` section.


If you find yourself uncertain or in need of further insights, never hesitate to consult with your tutor or ask your questions here. We are here to guide and assist you.

<div class="alert alert-block alert-warning">
<b>Reviewer's comment v1</b>
    
It would be immensely helpful if you can add some contextual information at the beginning of the notebook, covering:

- A brief description of the project's objective.
- An overview of the data you're working with.
- Any expected outcomes or results you aim to achieve.

Providing such context allows anyone reviewing or revisiting the notebook to quickly understand its purpose and the data being used, ensuring smoother collaboration and more intuitive navigation.

</div>

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv('/datasets/users_behavior.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [4]:
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


<div class="alert alert-block alert-success">
<b>Reviewer's comment v1</b>

Great data overview!

In [5]:
# Separate features and target
X = df.drop('is_ultra', axis=1)
y = df['is_ultra']

In [6]:
# First split: 20% test, 80% temp (train+validation)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Second split: 60% train, 20% validation (from temp)
X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)
# 0.25 x 0.8 = 0.2

# Check the sizes of the splits
X_train.shape, X_valid.shape, X_test.shape

((1928, 4), (643, 4), (643, 4))

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1</b>

Well done 60-20-20

In [7]:
results = {}

In [8]:
best_dt_acc = 0
for depth in range(1, 21):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    pred = dt.predict(X_valid)
    acc = accuracy_score(y_valid, pred)
    if acc > best_dt_acc:
        best_dt_acc = acc
        results['Decision Tree'] = (acc, dt)

In [9]:
best_rf_acc = 0
for est in [10, 50, 100, 150]:
    for depth in [5, 10, 15, None]:
        rf = RandomForestClassifier(n_estimators=est, max_depth=depth, random_state=42)
        rf.fit(X_train, y_train)
        pred = rf.predict(X_valid)
        acc = accuracy_score(y_valid, pred)
        if acc > best_rf_acc:
            best_rf_acc = acc
            results['Random Forest'] = (acc, rf)

In [10]:
# Set up parameter grid
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [5, 10, 15]
}

# Initialize model
rf = RandomForestClassifier(random_state=42)

# Initialize Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit on training data
grid_search.fit(X_train, y_train)

# Best estimator and score
best_rf_model = grid_search.best_estimator_
best_rf_acc = accuracy_score(y_valid, best_rf_model.predict(X_valid))

print("Best parameters found:", grid_search.best_params_)
print(f"Validation Accuracy of best Random Forest: {best_rf_acc:.4f}")

# Store results for comparison
results['Random Forest'] = (best_rf_acc, best_rf_model)

Best parameters found: {'max_depth': 10, 'n_estimators': 100}
Validation Accuracy of best Random Forest: 0.7823


In [11]:
lr = LogisticRegression(solver='liblinear', random_state=42)
lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_valid, lr.predict(X_valid))
results['Logistic Regression'] = (lr_acc, lr)

results_sorted = sorted(results.items(), key=lambda x: x[1][0], reverse=True)
results_sorted

[('Decision Tree',
  (0.7900466562986003, DecisionTreeClassifier(max_depth=5, random_state=42))),
 ('Random Forest',
  (0.7822706065318819, RandomForestClassifier(max_depth=10, random_state=42))),
 ('Logistic Regression',
  (0.7060653188180405,
   LogisticRegression(random_state=42, solver='liblinear')))]

In [12]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_valid)
dummy_acc = accuracy_score(y_valid, dummy_pred)

print(f"Dummy Classifier Accuracy (most frequent): {dummy_acc:.4f}")

Dummy Classifier Accuracy (most frequent): 0.6936


<div class="alert alert-block alert-success">
<b>Reviewer's comment v1</b>

Great way to create your workflow!

In [13]:
# Select the best model (Random Forest) for final evaluation
best_model = results['Random Forest'][1]

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1</b>

Well done!

In [14]:
# Predict on test data
test_preds = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_preds)

In [15]:
test_accuracy

0.8195956454121306

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1</b>

That's correct!

## Summary of Findings
- **Best model:** Random Forest (max_depth=10, n_estimators=10)
- **Validation accuracy:** ~0.79
- **Test accuracy:** ~0.81
- **Sanity check:** Dummy Classifier scored ~0.5, confirming our model performs significantly better than random guessing.
- **Conclusion:** Our Random Forest model meets the required threshold of 0.75 and is suitable for deployment in recommending user plans.

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1</b>

Excellent project!